In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

In [2]:
# DataFrame Info
# Columns:
# - Date:       Current date  -  Unique: 7465
# - Expiry:     Date of expiry  -> increases for newer dates                        [1996-01-04 - 2025-09-08]
# - Texp:       Time to expiry in years (difference between 'Date' and 'Expiry') number of expiries    [6, 528]
# - Strike:     Strike price
# - Bid:        Bid-price       -> has NA values ~9% -> Data seems clear enough to  ignore the NaN values
# - Ask:        Ask-price       
# - Fwd:        Forward-price
# - CallMid:    IV in bps?

In [3]:
df = pd.read_parquet("data/spxIvolList.parquet")
df.head()

,Expiry,Texp,Strike,Bid,Ask,Fwd,CallMid,Date
0,1996-01-20,0.043806,400.0,NaN,0.748694,616.91877,NaN,1996-01-04
1,1996-01-20,0.043806,525.0,NaN,0.331098,616.91877,NaN,1996-01-04
2,1996-01-20,0.043806,535.0,NaN,0.296718,616.91877,NaN,1996-01-04
3,1996-01-20,0.043806,540.0,0.256724,0.279628,616.91877,77.01841,1996-01-04
4,1996-01-20,0.043806,545.0,0.240831,0.262591,616.91877,72.01841,1996-01-04


In [4]:
def hasNotNaNInside(group):
    valueMask = ~np.isnan(group["w"])
    
    minIndex = np.argmax(valueMask)
    maxIndex = len(valueMask) - 1 - np.argmax(valueMask[::-1])

    return np.all(valueMask[minIndex:maxIndex + 1])

In [5]:
# Only consider data between 01/01/2000 and 31/12/2024
df = df[(df["Date"] >= pd.Timestamp("2000-01-01")) &
        (df["Date"] <= pd.Timestamp("2024-12-31"))]

# Calculate log moneyness and total implied variance
df["z"] = np.log(df["Strike"] / df["Fwd"])
df["w"] = (df["CallMid"] / 10_000) ** 2 * df["Texp"]

# Remove slices with NaN values inside
df = df[["Date", "Expiry", "Texp", "z", "w"]]
df = df.groupby(["Date", "Texp"]).filter(hasNotNaNInside)

In [6]:
# Group data into maturity bins.
bins = np.array([
    -np.inf, 
    7,
    30, 
    90,
    180, 
    365, 
    np.inf
    ])

labels= [
    "<=1wk",    # 0-7 days
    "1m",       # 7-30 days
    "3m",       # 30-90 days
    "6m",       # 90 - 180 days
    "1y",       # 180 - 365 days
    ">1y"       # 365+ days
]

# Calculate days to expiry
df["DTE"] =  (df["Expiry"] - df["Date"]).dt.days

# Group IV data into maturity groups
df["Group"] = pd.cut(
    df["DTE"],
    bins=list(bins),
    labels=labels,
    include_lowest=True
)

# Remove group ">1y" because of bad data
df = df[df["Group"] != ">1y"]

df.head()

,Date,Expiry,Texp,z,w,DTE,Group
219043,2000-01-03,2000-01-22,0.052019,-0.400915,NaN,19,1m
219044,2000-01-03,2000-01-22,0.052019,-0.350905,0.000097,19,1m
219045,2000-01-03,2000-01-22,0.052019,-0.326807,0.000086,19,1m
219046,2000-01-03,2000-01-22,0.052019,-0.303277,0.000076,19,1m
219047,2000-01-03,2000-01-22,0.052019,-0.280287,0.000066,19,1m


In [7]:
df.groupby(["Group"]).apply(lambda g: len(g.groupby(["Date", "Texp"])))

Group
<=1wk     4708
1m       22853
3m       22760
6m       17387
1y       23855
dtype: int64

In [8]:
df.to_parquet("data/ivdata.parquet")